# Task 3 — Parte A (ablação) e Parte B (regressão logística e árvore de decisão)

Este notebook cobre a **ablação por grupos de variáveis** e os **dois modelos individuais** desta parte do trabalho: **regressão logística** (linear) e **árvore de decisão**.

**Como funciona**

- Usa o `avaliar_folds` e a restante infraestrutura da Task 2 (`avaliacao.py`): 4 folds temporais, orçamento de 200 alertas por 1000 sessões, métrica principal = **recall@200**. O teste (out–dez/2021) é removido ao carregar os dados e nunca é usado aqui.
- O pré-processamento está em `ablacao.py` e segue as regras da Task 1 (imputação pela mediana, limite no percentil 99, log1p, escalonamento, one-hot). É **reajustado em cada fold**, só com o treino desse fold.
- Os hiperparâmetros são **fixos e iguais** em todas as combinações da ablação, para que as diferenças venham da informação e não do ajuste. A afinação vem depois (secção 7).

**Decisões que podem mudar (estão no bloco de configuração):** nível espacial do contexto (`ESPACO`) e hiperparâmetros. Se a colega decidir outra coisa, basta alterar esse bloco e voltar a correr.

| Secção | O que responde |
|---|---|
| 1 | Referência a superar (baselines) |
| 2 | Teste de sanidade: o circuito funciona? |
| 3 | Que nível espacial usar (posto, localização ou distrito)? |
| 4 | Ablação principal, números operacionais, tendência e elegibilidade |
| 5 | A tarifa acrescenta algo além da hora? |
| 6 | Robustez, grupo sem histórico e origem das falhas |
| 7 | Afinação dos hiperparâmetros e ponderação das classes |
| 8 | Configurações finais, Naive Bayes e comparação dos modelos individuais |
| 9 | Ensembles: diversidade, Random Forest, boosting e votação |
| 10 | Síntese: comparação final e escolha do modelo |

## 0. Configuração e dados

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

import ablacao as ab
from avaliacao import (avaliar, avaliar_folds, agregar_folds, comparar,
                       alertas_por_orcamento, folds_temporais, FOLDS)

warnings.filterwarnings("ignore", category=FutureWarning)   # os avisos de convergência continuam visíveis
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

DATA_PATH = Path("Charging_Data_educational.csv")

X_ALERTAS = 200            # orçamento de alertas por 1000 sessões (decidido na Task 2)
ESPACO = "posto"           # nível espacial no contexto: "posto" | "localizacao" | "distrito"
MODELOS = ["logistica", "arvore"]
print("Hiperparâmetros fixos:", ab.HIPERPARAMETROS)

import sklearn
SK_NOVO = tuple(int(x) for x in sklearn.__version__.split(".")[:2]) >= (1, 8)
print("sklearn", sklearn.__version__)

Hiperparâmetros fixos: {'logistica': {'C': 0.1, 'max_iter': 300}, 'arvore': {'max_depth': 8, 'min_samples_leaf': 200}, 'naive_bayes': {'var_smoothing': 1e-09}, 'random_forest': {'n_estimators': 200, 'min_samples_leaf': 100, 'max_features': 'sqrt', 'n_jobs': -1}}
sklearn 1.9.1


In [2]:
df = ab.carregar_dev(DATA_PATH)            # só treino e validação (dez/2019 – set/2021)
print(f"{len(df):,} sessões | {df['Start Time'].min():%Y-%m-%d} a {df['Start Time'].max():%Y-%m-%d} | prevalência {df['is_Abnormal'].mean():.1%}")
assert df["Start Time"].max() < pd.Timestamp("2021-10-01"), "o teste não pode estar carregado"

360,996 sessões | 2019-12-31 a 2021-09-30 | prevalência 18.8%


In [3]:
for m in MODELOS:
    cols, pipe = ab.criar_pipeline(m, ab.CONFIGURACOES["4 + histórico do utilizador"], ESPACO)
    pre = pipe.named_steps["pre"].fit(df[cols])
    print(m, "→", pre.transform(df[cols].head()).shape[1], "colunas")

logistica → 151 colunas
arvore → 122 colunas


## 1. Referência a superar

A Task 2 fixou, como referência, a regra **«a última sessão concluída do posto foi anómala»** (recall@200 médio de 0,511) e a taxa de anomalias do posto a 7 dias. Recalculam-se aqui com os mesmos folds, para confirmar que o circuito reproduz os valores da Task 2.

In [4]:
def metricas_fold(res):
    return pd.Series({"recall@200 (média)": res["recall"].mean(), "desvio": res["recall"].std(ddof=1),
                      "pior fold": res["recall"].min(),
                      "PR-AUC norm. (média)": res["pr_auc_norm"].mean(), "ROC-AUC (média)": res["roc_auc"].mean()})

baselines = {
    "regra: última sessão do posto anómala": avaliar_folds(df, lambda tr, va: va["post_last_completed_abnormal"].to_numpy(), X_ALERTAS),
    "taxa de anomalias do posto (7 dias)":   avaliar_folds(df, lambda tr, va: va["post_previous_abnormal_rate_7D"].to_numpy(), X_ALERTAS),
}
ref = pd.DataFrame({k: metricas_fold(v) for k, v in baselines.items()}).T.round(3)
ref

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
regra: última sessão do posto anómala,0.511,0.020,0.482,0.201,0.698
taxa de anomalias do posto (7 dias),0.499,0.024,0.470,0.404,0.772


In [5]:
ref_regra_res, ref_regra_sc = avaliar_folds(
    df, lambda tr, va: va["post_last_completed_abnormal"].to_numpy(), X_ALERTAS, devolver_scores=True)
ref_pr = baselines["taxa de anomalias do posto (7 dias)"]

## 2. Teste de sanidade

Antes das 8 corridas da ablação, corre-se **cada modelo, com todas as variáveis** (combinação 4), nos 4 folds. O que se espera:

- Melhor do que a referência, mas **sem saltos enormes**: o sinal vem sobretudo dos históricos, que a regra da última sessão já capta.
- Um resultado muito acima (por exemplo, recall perto de 0,9) seria sinal de **fuga de informação**: verificar se entrou alguma variável pós-sessão ou se algo foi ajustado com dados de validação.

In [6]:
GRUPOS_TODOS = ab.CONFIGURACOES["4 + histórico do utilizador"]
sanidade = {}
for m in MODELOS:
    sanidade[m] = avaliar_folds(df, ab.fabricar_scorer(m, GRUPOS_TODOS, ESPACO), X_ALERTAS, devolver_scores=True)

linhas = {m: metricas_fold(r[0]) for m, r in sanidade.items()}
linhas["(referência) última sessão do posto"] = metricas_fold(baselines["regra: última sessão do posto anómala"])
pd.DataFrame(linhas).T.round(3)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
logistica,0.580,0.015,0.566,0.521,0.831
arvore,0.575,0.012,0.563,0.522,0.827
(referência) última sessão do posto,0.511,0.020,0.482,0.201,0.698


In [7]:
# recall@200 por fold: modelos vs referência
ref_regra = baselines["regra: última sessão do posto anómala"]["recall"]
por_fold = pd.DataFrame({**{m: r[0]["recall"] for m, r in sanidade.items()}, "referência": ref_regra}).round(3)
por_fold["logística − referência"] = (por_fold["logistica"] - por_fold["referência"]).round(3)
por_fold["árvore − referência"] = (por_fold["arvore"] - por_fold["referência"]).round(3)
por_fold

,logistica,arvore,referência,logística − referência,árvore − referência
F1,0.590,0.586,0.520,0.070,0.066
F2,0.568,0.563,0.482,0.086,0.081
F3,0.594,0.585,0.526,0.068,0.059
F4,0.566,0.568,0.518,0.048,0.050


## 3. Nível espacial no contexto

A Task 1 mostrou que posto, localização e distrito são redundantes entre si (cada posto determina os outros dois), e a secção 8.2 do relatório prevê compará-los como alternativas. Compara-se com a **árvore** (rápida) e com todas as variáveis, incluindo também a opção **sem informação espacial**. Com todas as variáveis as opções empatam pela regra da secção 8.9, por isso compara-se também a logística na combinação 1 (sem históricos), onde o nível espacial faz diferença.

In [8]:
espaciais = {}
for esp in ["posto", "localizacao", "distrito", None]:
    espaciais[esp or "sem espaço"] = avaliar_folds(df, ab.fabricar_scorer("arvore", GRUPOS_TODOS, esp), X_ALERTAS)

tab_esp = pd.DataFrame({k: metricas_fold(v) for k, v in espaciais.items()}).T.round(3)
tab_esp

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
posto,0.575,0.012,0.563,0.522,0.827
localizacao,0.576,0.011,0.564,0.522,0.826
distrito,0.577,0.013,0.562,0.522,0.827
sem espaço,0.576,0.013,0.563,0.522,0.827


In [9]:
# comparação emparelhada com o posto (A = alternativa, B = posto)
base = espaciais["posto"]
pd.DataFrame({k: comparar(v, base, "recall") for k, v in espaciais.items() if k != "posto"}).T.round(3)

,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B
localizacao,0.000,0.001,1.0,0.564,0.563
distrito,0.001,0.002,2.0,0.562,0.563
sem espaço,0.001,0.002,1.0,0.563,0.563


In [10]:
esp_c1 = {}
for esp in ["posto", "localizacao", "distrito", None]:
    esp_c1[esp or "sem espaço"] = avaliar_folds(
        df, ab.fabricar_scorer("logistica", ab.CONFIGURACOES["1 contexto + meteorologia"], esp), X_ALERTAS)
pd.DataFrame({k: metricas_fold(v) for k, v in esp_c1.items()}).T.round(3)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
posto,0.375,0.018,0.354,0.177,0.710
localizacao,0.337,0.007,0.331,0.132,0.677
distrito,0.293,0.017,0.273,0.070,0.598
sem espaço,0.219,0.011,0.205,0.016,0.524


## 4. Ablação principal

Quatro combinações **acumuladas**, avaliadas com os dois modelos, nos 4 folds:

1. contexto (tempo + nível espacial) + meteorologia
2. \+ tarifa
3. \+ histórico do posto
4. \+ histórico do utilizador

Na regressão logística, a hora e o dia da semana entram em **one-hot**, para o modelo conseguir reproduzir qualquer padrão horário. Só assim a pergunta «a tarifa acrescenta algo além da hora?» tem resposta honesta (o preço é função da hora). Na árvore, entram como números.

In [11]:
ablacao = {}     # (configuração, modelo) -> (resultados por fold, scores)
for nome_cfg, grupos in ab.CONFIGURACOES.items():
    for m in MODELOS:
        ablacao[(nome_cfg, m)] = avaliar_folds(df, ab.fabricar_scorer(m, grupos, ESPACO), X_ALERTAS, devolver_scores=True)

linhas = {}
for (cfg, m), (res, _) in ablacao.items():
    linhas[(m, cfg)] = metricas_fold(res)
tab_abl = pd.DataFrame(linhas).T.round(3)
tab_abl

,,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
logistica,1 contexto + meteorologia,0.375,0.018,0.354,0.177,0.710
arvore,1 contexto + meteorologia,0.275,0.031,0.232,0.054,0.551
logistica,2 + tarifa,0.375,0.018,0.354,0.177,0.710
arvore,2 + tarifa,0.275,0.030,0.234,0.054,0.552
logistica,3 + histórico do posto,0.550,0.015,0.529,0.491,0.804
arvore,3 + histórico do posto,0.557,0.017,0.534,0.491,0.807
logistica,4 + histórico do utilizador,0.580,0.015,0.566,0.521,0.831
arvore,4 + histórico do utilizador,0.575,0.012,0.563,0.522,0.827


In [12]:
ops = ["precisao", "alertas_corretos_1000", "falsos_alertas_1000", "falhadas_1000"]
linhas = {("regra", "referência (última sessão)"): ref_regra_res[ops].mean()}
for (cfg, m), (res, _) in ablacao.items():
    linhas[(m, cfg)] = res[ops].mean()
pd.DataFrame(linhas).T.round(3)

,,precisao,alertas_corretos_1000,falsos_alertas_1000,falhadas_1000
regra,referência (última sessão),0.461,92.281,107.716,87.776
logistica,1 contexto + meteorologia,0.337,67.475,132.522,112.581
arvore,1 contexto + meteorologia,0.249,49.851,150.147,130.206
logistica,2 + tarifa,0.337,67.480,132.517,112.577
arvore,2 + tarifa,0.250,49.934,150.064,130.123
logistica,3 + histórico do posto,0.495,99.087,100.910,80.969
arvore,3 + histórico do posto,0.502,100.479,99.519,79.578
logistica,4 + histórico do utilizador,0.521,104.286,95.711,75.770
arvore,4 + histórico do utilizador,0.518,103.595,96.402,76.461


In [13]:
# recall@200 por fold, para ver a consistência no tempo
por_fold = pd.DataFrame({(m, cfg): res["recall"] for (cfg, m), (res, _) in ablacao.items()}).round(3)
por_fold.columns = pd.MultiIndex.from_tuples(por_fold.columns, names=["modelo", "configuração"])
por_fold.T

,,F1,F2,F3,F4
modelo,configuração,,,,
logistica,1 contexto + meteorologia,0.354,0.398,0.376,0.373
arvore,1 contexto + meteorologia,0.232,0.278,0.287,0.303
logistica,2 + tarifa,0.354,0.398,0.379,0.370
arvore,2 + tarifa,0.234,0.278,0.287,0.303
logistica,3 + histórico do posto,0.553,0.529,0.563,0.553
arvore,3 + histórico do posto,0.560,0.534,0.574,0.561
logistica,4 + histórico do utilizador,0.590,0.568,0.594,0.566
arvore,4 + histórico do utilizador,0.586,0.563,0.585,0.568


In [14]:
# efeito de cada grupo acrescentado: configuração seguinte vs anterior (diferenças emparelhadas, fold a fold)
nomes = list(ab.CONFIGURACOES)
linhas = {}
for m in MODELOS:
    for ant, seg in zip(nomes[:-1], nomes[1:]):
        c = comparar(ablacao[(seg, m)][0], ablacao[(ant, m)][0], "recall")
        linhas[(m, f"{seg}  vs  {ant}")] = c
efeitos = pd.DataFrame(linhas).T
efeitos[["diferença média", "desvio da diferença"]] = efeitos[["diferença média", "desvio da diferença"]].astype(float).round(4)
efeitos[["diferença média", "desvio da diferença", "folds em que A ganha"]]

diferença média  desvio da diferença  folds em que A ganha
logistica 2 + tarifa  vs  1 contexto + meteorologia                    0.0001               0.0020                   1.0
          3 + histórico do posto  vs  2 + tarifa                       0.1742               0.0298                   4.0
          4 + histórico do utilizador  vs  3 + histórico ...           0.0300               0.0120                   4.0
arvore    2 + tarifa  vs  1 contexto + meteorologia                    0.0005               0.0010                   1.0
          3 + histórico do posto  vs  2 + tarifa                       0.2817               0.0333                   4.0
          4 + histórico do utilizador  vs  3 + histórico ...           0.0182               0.0105                   4.0

In [15]:
tend = {m: agregar_folds(*ablacao[("4 + histórico do utilizador", m)]) for m in MODELOS}
tend["referência (última sessão)"] = agregar_folds(ref_regra_res, ref_regra_sc)
tend["taxa do posto (7 dias)"] = agregar_folds(ref_pr)
pd.concat(tend).round(3)

média  desvio  pior fold fold do pior  média ponderada  tendência por fold  pooled
logistica                  recall       0.580   0.015      0.566           F4            0.579              -0.005   0.579
                           pr_auc       0.607   0.043      0.549           F2            0.614               0.021   0.618
                           pr_auc_norm  0.521   0.040      0.464           F2            0.527               0.017   0.532
                           roc_auc      0.831   0.008      0.820           F2            0.832               0.002   0.833
                           precisao     0.521   0.063      0.449           F2            0.532               0.040   0.532
arvore                     recall       0.575   0.012      0.563           F2            0.576              -0.003   0.576
                           pr_auc       0.607   0.043      0.551           F2            0.614               0.024   0.623
                           pr_auc_norm  0.522   0.041      0.467           F2            0.528               0.020   0.538
                           roc_auc      0.827   0.009      0.814           F2            0.828               0.004   0.830
                           precisao     0.518   0.066      0.445           F2            0.529               0.041   0.529
referência (última sessão) recall       0.511   0.020      0.482           F2            0.514               0.004   0.515
                           pr_auc       0.344   0.042      0.294           F2            0.351               0.024   0.354
                           pr_auc_norm  0.201   0.030      0.161           F2            0.206               0.014   0.208
                           roc_auc      0.698   0.015      0.677           F2            0.701               0.006   0.702
                           precisao     0.461   0.070      0.381           F2            0.473               0.043   0.473
taxa do posto (7 dias)     recall       0.499   0.024      0.470           F2            0.502               0.011     NaN
                           pr_auc       0.511   0.042      0.456           F2            0.516               0.021     NaN
                           pr_auc_norm  0.404   0.038      0.354           F2            0.408               0.015     NaN
                           roc_auc      0.772   0.020      0.749           F2            0.775               0.011     NaN
                           precisao     0.451   0.073      0.372           F2            0.462               0.048     NaN

In [16]:
eleg = {}
for m in MODELOS:
    r = ablacao[("4 + histórico do utilizador", m)][0]
    t = pd.DataFrame({"recall modelo": r["recall"], "recall regra": ref_regra_res["recall"],
                      "PR-AUC norm. modelo": r["pr_auc_norm"], "PR-AUC norm. taxa 7d": ref_pr["pr_auc_norm"]})
    t["elegível"] = (t["recall modelo"] > t["recall regra"]) & (t["PR-AUC norm. modelo"] > t["PR-AUC norm. taxa 7d"])
    eleg[m] = t
pd.concat(eleg).round(3)

recall modelo  recall regra  PR-AUC norm. modelo  PR-AUC norm. taxa 7d  elegível
logistica F1          0.590         0.520                0.522                 0.398      True
          F2          0.568         0.482                0.464                 0.354      True
          F3          0.594         0.526                0.547                 0.446      True
          F4          0.566         0.518                0.551                 0.417      True
arvore    F1          0.586         0.520                0.516                 0.398      True
          F2          0.563         0.482                0.467                 0.354      True
          F3          0.585         0.526                0.548                 0.446      True
          F4          0.568         0.518                0.557                 0.417      True

## 5. A tarifa acrescenta algo além da hora?

O preço é função da hora, por isso a tarifa só pode trazer informação nova se o modelo não conseguir já reproduzir o padrão horário. Duas verificações:

- **Com a hora** (secção 4, combinações 1 vs 2): a diferença deve ser ≈ 0.
- **Sem a hora**: se a tarifa substituir a hora, recupera-se parte do efeito da hora? Isto mostra que o preço é uma versão grosseira da hora.

In [17]:
sem_hora = {
    "meteorologia":                 ["meteorologia"],
    "meteorologia + tarifa":        ["meteorologia", "tarifa"],
    "meteorologia + hora/dia":      ["tempo", "meteorologia"],
    "meteorologia + hora/dia + tarifa": ["tempo", "meteorologia", "tarifa"],
}
linhas = {}
for nome_cfg, grupos in sem_hora.items():
    for m in MODELOS:
        r = avaliar_folds(df, ab.fabricar_scorer(m, grupos, ESPACO), X_ALERTAS)
        linhas[(m, nome_cfg)] = metricas_fold(r)
pd.DataFrame(linhas).T.round(3)

,,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
logistica,meteorologia,0.372,0.022,0.347,0.178,0.710
arvore,meteorologia,0.276,0.031,0.232,0.054,0.552
logistica,meteorologia + tarifa,0.371,0.021,0.348,0.177,0.710
arvore,meteorologia + tarifa,0.276,0.030,0.234,0.054,0.553
logistica,meteorologia + hora/dia,0.375,0.018,0.354,0.177,0.710
arvore,meteorologia + hora/dia,0.275,0.031,0.232,0.054,0.551
logistica,meteorologia + hora/dia + tarifa,0.375,0.018,0.354,0.177,0.710
arvore,meteorologia + hora/dia + tarifa,0.275,0.030,0.234,0.054,0.552


## 6. Análises de robustez prometidas no relatório

A secção 8.10 (e a 7.5) do relatório comprometem-se a reportar estas análises. Usam a **combinação 4** com os dois modelos.

In [18]:
# ---------- 6.1 Sensibilidade ao orçamento X
cfg4 = {m: ablacao[("4 + histórico do utilizador", m)][1] for m in MODELOS}      # scores por fold
sc_base = baselines["regra: última sessão do posto anómala"]

def recall_com_X(scores, X):
    return np.mean([avaliar(y, s, pred=alertas_por_orcamento(s, X))["recall"] for y, s, _ in scores.values()])

linhas = {}
for X in [100, 150, 200, 300]:
    linhas[X] = {m: recall_com_X(cfg4[m], X) for m in MODELOS}
    # referência com o mesmo orçamento
    ref_sc = avaliar_folds(df, lambda tr, va: va["post_last_completed_abnormal"].to_numpy(), X)["recall"].mean()
    linhas[X]["referência (última sessão)"] = ref_sc
sens = pd.DataFrame(linhas).T.round(3); sens.index.name = "alertas por 1000"
sens

,logistica,arvore,referência (última sessão)
alertas por 1000,,,
100,0.395,0.399,0.282
150,0.500,0.501,0.423
200,0.580,0.575,0.511
300,0.700,0.694,0.578


In [19]:
# ---------- 6.2 Sessões sem histórico do utilizador e sem as contas 39 e 23
# Recall dentro de cada subconjunto, mantendo os alertas decididos com o orçamento global de cada fold
def recall_subconjunto(scores, mascara):
    r = {}
    for nome, tr, va in folds_temporais(df["Start Time"]):
        y, s, pred = scores[nome]
        m = mascara[va]
        r[nome] = (pred[m][y[m] == 1].sum() / max((y[m] == 1).sum(), 1))
    return pd.Series(r)

mask_sem_hist = (df["user_no_previous_completed_session"] == 1).to_numpy()
mask_sem_39_23 = (~df["UserID"].isin([39, 23])).to_numpy()
mask_com_hist = ~mask_sem_hist

linhas = {}
for m in MODELOS:
    linhas[(m, "todas as sessões")] = sanidade[m][0]["recall"]
    linhas[(m, "só sem histórico do utilizador")] = recall_subconjunto(cfg4[m], mask_sem_hist)
    linhas[(m, "só com histórico do utilizador")] = recall_subconjunto(cfg4[m], mask_com_hist)
    linhas[(m, "sem as contas 39 e 23")] = recall_subconjunto(cfg4[m], mask_sem_39_23)
robust = pd.DataFrame(linhas).T
robust["média"] = robust.mean(axis=1)
robust.round(3)

F1     F2     F3     F4  média
logistica todas as sessões                0.590  0.568  0.594  0.566  0.580
          só sem histórico do utilizador  0.427  0.426  0.483  0.490  0.456
          só com histórico do utilizador  0.617  0.595  0.617  0.579  0.602
          sem as contas 39 e 23           0.593  0.571  0.596  0.568  0.582
arvore    todas as sessões                0.586  0.563  0.585  0.568  0.575
          só sem histórico do utilizador  0.367  0.349  0.397  0.482  0.399
          só com histórico do utilizador  0.622  0.603  0.623  0.584  0.608
          sem as contas 39 e 23           0.589  0.566  0.587  0.571  0.578

In [20]:
linhas = {}
for nome_sub, mascara in {"sem histórico do utilizador": mask_sem_hist, "com histórico do utilizador": mask_com_hist}.items():
    linhas[("referência", nome_sub)] = recall_subconjunto(ref_regra_sc, mascara)
rr = pd.DataFrame(linhas).T
rr["média"] = rr.mean(axis=1)
display(rr.round(3))

linhas = {}
for nome, tr, va in folds_temporais(df["Start Time"]):
    y = df["is_Abnormal"].to_numpy()[va]
    m = mask_sem_hist[va]
    linha = {"% sessões sem histórico": m.mean(),
             "prevalência sem histórico": y[m].mean(),
             "prevalência com histórico": y[~m].mean()}
    for mod in MODELOS:
        pred = np.asarray(cfg4[mod][nome][2])
        linha[f"% alertas p/ sem histórico ({mod})"] = pred[m].sum() / pred.sum()
    linhas[nome] = linha
pd.DataFrame(linhas).T.round(3)

F1     F2     F3     F4  média
referência sem histórico do utilizador  0.379  0.332  0.423  0.423  0.389
           com histórico do utilizador  0.543  0.511  0.547  0.535  0.534

,% sessões sem histórico,prevalência sem histórico,prevalência com histórico,% alertas p/ sem histórico (logistica),% alertas p/ sem histórico (arvore)
F1,0.113,0.210,0.163,0.124,0.096
F2,0.125,0.202,0.152,0.162,0.108
F3,0.142,0.217,0.177,0.158,0.105
F4,0.128,0.253,0.205,0.143,0.137


In [21]:
# ---------- 6.3 Com e sem o período de arranque no treino (treino a partir de 1 de abril de 2020)
linhas = {}
for m in MODELOS:
    com = ablacao[("4 + histórico do utilizador", m)][0]
    sem = avaliar_folds(df, ab.fabricar_scorer(m, GRUPOS_TODOS, ESPACO, treino_a_partir_de="2020-04-01"), X_ALERTAS)
    linhas[(m, "treino completo")] = com["recall"]
    linhas[(m, "sem o arranque (treino a partir de abr/2020)")] = sem["recall"]
arranque = pd.DataFrame(linhas).T
arranque["média"] = arranque.mean(axis=1)
arranque.round(3)

F1     F2     F3     F4  média
logistica treino completo                               0.590  0.568  0.594  0.566  0.580
          sem o arranque (treino a partir de abr/2020)  0.587  0.568  0.594  0.566  0.579
arvore    treino completo                               0.586  0.563  0.585  0.568  0.575
          sem o arranque (treino a partir de abr/2020)  0.588  0.561  0.589  0.561  0.575

### 6.4 Origem das falhas (equipamento vs veículo)

In [22]:
ORIGEM = {
    "Other faults of the charging pile": "equipamento",
    "DC Output Fault": "equipamento",
    "Faulty charging connection": "equipamento",
    "Failure of non-system control loop components and assemblies": "equipamento",
    "Power conversion unit failure": "equipamento",
    "DC switching device failure": "equipamento",
    "System communications failure": "equipamento",
    "AC Input Failure": "equipamento",
    "AC switching device failure": "equipamento",
    "Vehicle communications failure": "veículo",
    "BMS failure": "veículo",
    "Vehicle charging faults": "veículo",
    "Vehicle Stake Interaction Failure": "interação posto-veículo",
}
an = df.loc[df["is_Abnormal"] == 1, "end_cause"]
assert set(an.unique()) <= set(ORIGEM), set(an.unique()) - set(ORIGEM)
display(an.value_counts().rename("sessões").to_frame().assign(origem=lambda t: t.index.map(ORIGEM)))
an.map(ORIGEM).value_counts(normalize=True).round(3)

,sessões,origem
end_cause,,
Other faults of the charging pile,16917,equipamento
DC Output Fault,12587,equipamento
Vehicle communications failure,11620,veículo
Faulty charging connection,7014,equipamento
BMS failure,6584,veículo
Failure of non-system control loop components and assemblies,6123,equipamento
Vehicle charging faults,4768,veículo
Power conversion unit failure,1394,equipamento
DC switching device failure,591,equipamento


end_cause
equipamento                0.661
veículo                    0.339
interação posto-veículo    0.000
Name: proportion, dtype: float64

## 7. Afinação dos hiperparâmetros

Só depois de fechada a ablação. Escolhem-se, **na combinação final** (todas as variáveis, posto), os hiperparâmetros de cada modelo pela regra da secção 8.9 para hiperparâmetros: ordenação pelo recall médio; diferenças inferiores a 0,005 são empate e são desempatadas pela PR-AUC normalizada. Mostra-se também a comparação emparelhada, fold a fold, de cada candidato contra a configuração fixa usada na ablação.

| Modelo | Hiperparâmetros a variar | Grelha |
|---|---|---|
| Regressão logística | `C` (regularização) | 0,01 · 0,1 · 1 · 10 (penalização L2) |
| Árvore de decisão | `max_depth`, `min_samples_leaf` | 4 · 6 · 8 · 12 × 50 · 200 · 1000 |

Cuidados:

- A afinação **usa os mesmos folds** que a comparação, por isso o ganho medido é ligeiramente otimista. Por isso, diferenças de recall inferiores a 0,005 são tratadas como empate.
- O **teste continua intocado** até ao fim.
- A penalização **L1** foi testada à parte (C de 0,01 a 10) e deu o mesmo resultado que a L2 (recall 0,579 a 0,580), mas demora cerca de 15 minutos. Fica desligada por omissão (`INCLUIR_L1 = False`).

In [23]:
GRUPOS_TODOS = ab.CONFIGURACOES["4 + histórico do utilizador"]
INCLUIR_L1 = False          # True acrescenta a penalização L1 (muito mais lento, resultado igual)

GRELHAS = {
    "logistica": [{"C": c} for c in (0.01, 0.1, 1, 10)],
    "arvore":    [{"max_depth": d, "min_samples_leaf": l} for d in (4, 6, 8, 12) for l in (50, 200, 1000)],
}
if INCLUIR_L1:
        L1 = {"l1_ratio": 1} if SK_NOVO else {"penalty": "l1"}
        GRELHAS["logistica"] += [{"C": c, "solver": "liblinear", **L1} for c in (0.01, 0.1, 1, 10)]

def nome_hp(hp):
    return ", ".join(f"{k}={v}" for k, v in hp.items() if k != "solver")

def afinar(modelo):
    """Corre a grelha do modelo nos 4 folds. Devolve {nome: (hiperparâmetros, resultados por fold)}."""
    return {nome_hp(hp): (hp, avaliar_folds(df, ab.fabricar_scorer(modelo, GRUPOS_TODOS, ESPACO, **hp), X_ALERTAS))
            for hp in GRELHAS[modelo]}

afinacao = {}

In [24]:
# Árvore de decisão
afinacao["arvore"] = afinar("arvore")
pd.DataFrame({k: metricas_fold(r) for k, (hp, r) in afinacao["arvore"].items()}).T \
  .sort_values("recall@200 (média)", ascending=False).round(3)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
"max_depth=12, min_samples_leaf=200",0.578,0.013,0.564,0.525,0.823
"max_depth=8, min_samples_leaf=1000",0.578,0.015,0.563,0.517,0.827
"max_depth=12, min_samples_leaf=1000",0.577,0.014,0.563,0.520,0.826
"max_depth=8, min_samples_leaf=50",0.577,0.014,0.564,0.518,0.825
"max_depth=6, min_samples_leaf=1000",0.576,0.013,0.563,0.507,0.823
"max_depth=8, min_samples_leaf=200",0.575,0.012,0.563,0.522,0.827
"max_depth=6, min_samples_leaf=50",0.575,0.016,0.561,0.509,0.823
"max_depth=6, min_samples_leaf=200",0.575,0.014,0.562,0.509,0.823
"max_depth=4, min_samples_leaf=1000",0.566,0.013,0.552,0.470,0.808
"max_depth=4, min_samples_leaf=200",0.563,0.012,0.551,0.468,0.808


In [25]:
fixa = afinacao["arvore"]["max_depth=8, min_samples_leaf=200"][1]
pd.DataFrame({k: comparar(r, fixa, "recall") for k, (hp, r) in afinacao["arvore"].items()
              if k != "max_depth=8, min_samples_leaf=200"}).T.sort_values("diferença média", ascending=False).round(4)

,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B
"max_depth=12, min_samples_leaf=200",0.0026,0.0014,4.0,0.5639,0.5625
"max_depth=8, min_samples_leaf=1000",0.0024,0.0060,3.0,0.5633,0.5625
"max_depth=12, min_samples_leaf=1000",0.0020,0.0062,2.0,0.5633,0.5625
"max_depth=8, min_samples_leaf=50",0.0012,0.0035,2.0,0.5638,0.5625
"max_depth=6, min_samples_leaf=1000",0.0003,0.0051,2.0,0.5633,0.5625
"max_depth=6, min_samples_leaf=50",-0.0007,0.0044,2.0,0.5606,0.5625
"max_depth=6, min_samples_leaf=200",-0.0009,0.0036,2.0,0.5618,0.5625
"max_depth=4, min_samples_leaf=1000",-0.0097,0.0033,0.0,0.5521,0.5625
"max_depth=4, min_samples_leaf=50",-0.0124,0.0042,0.0,0.5507,0.5625
"max_depth=4, min_samples_leaf=200",-0.0124,0.0042,0.0,0.5507,0.5625


In [26]:
# Regressão logística
afinacao["logistica"] = afinar("logistica")
pd.DataFrame({k: metricas_fold(r) for k, (hp, r) in afinacao["logistica"].items()}).T \
  .sort_values("recall@200 (média)", ascending=False).round(3)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
C=1,0.58,0.015,0.565,0.521,0.831
C=10,0.58,0.015,0.566,0.521,0.831
C=0.01,0.58,0.014,0.566,0.521,0.831
C=0.1,0.58,0.015,0.566,0.521,0.831


### 7.1 Ponderação das classes

In [27]:
pond = {m: avaliar_folds(df, ab.fabricar_scorer(m, GRUPOS_TODOS, ESPACO, class_weight="balanced"), X_ALERTAS)
        for m in MODELOS}
pd.DataFrame({**{f"{m} (balanced)": metricas_fold(r) for m, r in pond.items()},
              **{m: metricas_fold(ablacao[("4 + histórico do utilizador", m)][0]) for m in MODELOS}}).T.round(3)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
logistica (balanced),0.579,0.014,0.565,0.517,0.831
arvore (balanced),0.576,0.018,0.556,0.523,0.827
logistica,0.580,0.015,0.566,0.521,0.831
arvore,0.575,0.012,0.563,0.522,0.827


In [28]:
pd.DataFrame({m: comparar(pond[m], ablacao[("4 + histórico do utilizador", m)][0], "recall") for m in MODELOS}).T.round(4)

,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B
logistica,-0.0003,0.0011,1.0,0.5649,0.5657
arvore,0.0006,0.0060,2.0,0.5563,0.5625


## 8. Modelos individuais: configurações finais e Naive Bayes

Os três modelos são corridos de novo com a configuração final (logística com C = 0,1; árvore com profundidade 12 e folha mínima 200; Naive Bayes gaussiano), na combinação 4. Guardam-se os scores, para os ensembles, e o tempo de cada corrida nos 4 folds. O Naive Bayes é testado com e sem o posto.

In [29]:
import time

resultados, scores, tempos = {}, {}, {}

def correr(nome_res, modelo, espaco, **hp):
    t0 = time.perf_counter()
    resultados[nome_res], scores[nome_res] = avaliar_folds(
        df, ab.fabricar_scorer(modelo, GRUPOS_TODOS, espaco, **hp), X_ALERTAS, devolver_scores=True)
    tempos[nome_res] = time.perf_counter() - t0

correr("logistica", "logistica", "posto")
correr("arvore", "arvore", "posto", max_depth=12, min_samples_leaf=200)
correr("naive_bayes (sem posto)", "naive_bayes", None)
correr("naive_bayes (com posto)", "naive_bayes", "posto")

def tabela(nomes):
    t = pd.DataFrame({n: metricas_fold(resultados[n]) for n in nomes}).T
    t["tempo 4 folds (s)"] = [tempos[n] for n in nomes]
    return t.round(3)

tabela(list(resultados))

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média),tempo 4 folds (s)
logistica,0.580,0.015,0.566,0.521,0.831,20.983
arvore,0.578,0.013,0.564,0.525,0.823,24.267
naive_bayes (sem posto),0.574,0.017,0.559,0.498,0.814,2.269
naive_bayes (com posto),0.492,0.017,0.477,0.276,0.757,4.378


In [30]:
display(pd.DataFrame({**{n: r["recall"] for n, r in resultados.items()},
                      "ref. última sessão": ref_regra_res["recall"]}).round(3))
pd.DataFrame({**{n: r["pr_auc_norm"] for n, r in resultados.items()},
              "ref. taxa 7 dias": ref_pr["pr_auc_norm"]}).round(3)

,logistica,arvore,naive_bayes (sem posto),naive_bayes (com posto),ref. última sessão
F1,0.590,0.590,0.586,0.477,0.520
F2,0.568,0.564,0.561,0.489,0.482
F3,0.594,0.587,0.592,0.517,0.526
F4,0.566,0.570,0.559,0.485,0.518


,logistica,arvore,naive_bayes (sem posto),naive_bayes (com posto),ref. taxa 7 dias
F1,0.522,0.517,0.505,0.254,0.398
F2,0.464,0.471,0.454,0.258,0.354
F3,0.547,0.550,0.522,0.301,0.446
F4,0.551,0.563,0.510,0.293,0.417


### 8.1 Afinação do Naive Bayes

Com o posto em one-hot, o Naive Bayes perde muito (recall 0,492, contra 0,574 sem posto): as 92 colunas binárias são tratadas como gaussianas e independentes, e o seu peso conjunto domina a decisão. Como o posto não acrescenta informação quando os históricos estão presentes (secção 3), o Naive Bayes usa a combinação 4 sem informação espacial. Afina-se o `var_smoothing`, que soma a todas as variâncias uma fração da maior variância e torna o modelo menos sensível a variáveis com variância pequena.

In [31]:
afin_nb = {}
for vs in (1e-9, 1e-6, 1e-3, 1e-2, 1e-1, 1.0):
    afin_nb[f"var_smoothing={vs:g}"] = avaliar_folds(
        df, ab.fabricar_scorer("naive_bayes", GRUPOS_TODOS, None, var_smoothing=vs), X_ALERTAS)
display(pd.DataFrame({k: metricas_fold(r) for k, r in afin_nb.items()}).T.round(3))

base_nb = afin_nb["var_smoothing=1e-09"]
pd.DataFrame({k: comparar(r, base_nb, "recall") for k, r in afin_nb.items()
              if k != "var_smoothing=1e-09"}).T.round(4)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
var_smoothing=1e-09,0.574,0.017,0.559,0.498,0.814
var_smoothing=1e-06,0.574,0.017,0.559,0.498,0.814
var_smoothing=0.001,0.574,0.017,0.559,0.498,0.814
var_smoothing=0.01,0.574,0.017,0.558,0.498,0.814
var_smoothing=0.1,0.572,0.018,0.554,0.497,0.814
var_smoothing=1,0.566,0.017,0.550,0.493,0.815


,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B
var_smoothing=1e-06,0.0000,0.0000,0.0,0.5593,0.5593
var_smoothing=0.001,-0.0001,0.0002,1.0,0.5593,0.5593
var_smoothing=0.01,-0.0006,0.0007,1.0,0.5584,0.5593
var_smoothing=0.1,-0.0029,0.0016,0.0,0.5542,0.5593
var_smoothing=1,-0.0086,0.0009,0.0,0.5504,0.5593


### 8.2 Comparação dos três modelos individuais (regra da secção 8.9)

In [32]:
nomes = ["logistica", "arvore", "naive_bayes (sem posto)"]
linhas = {}
for a in nomes:
    for b in nomes:
        if a != b:
            linhas[(a, b)] = comparar(resultados[a], resultados[b], "recall")
pd.DataFrame(linhas).T.round(4)

diferença média  desvio da diferença  folds em que A ganha  pior fold A  pior fold B
logistica               arvore                            0.0015               0.0050                   2.0       0.5657       0.5639
                        naive_bayes (sem posto)           0.0051               0.0021                   4.0       0.5657       0.5593
arvore                  logistica                        -0.0015               0.0050                   2.0       0.5639       0.5657
                        naive_bayes (sem posto)           0.0036               0.0062                   3.0       0.5639       0.5593
naive_bayes (sem posto) logistica                        -0.0051               0.0021                   0.0       0.5593       0.5657
                        arvore                           -0.0036               0.0062                   1.0       0.5593       0.5639

**Decisão.** O `var_smoothing` não altera o resultado até 0,01 e piora a partir de 0,1, pelo que se mantém o valor por omissão (1e-9). Pela regra da secção 8.9, a logística e a árvore são equivalentes (a logística tem média superior, +0,0015, mas só ganha em 2 de 4 folds); o desempate pelo menor desvio entre folds prefere a árvore (0,013 contra 0,015), que tem também maior PR-AUC normalizada (0,525 contra 0,521). Ambas são preferidas ao Naive Bayes: a logística ganha-lhe em 4 folds e a árvore em 3, com melhor pior fold.

### 8.3 Interpretabilidade

Os dois modelos interpretáveis são ajustados uma vez com o treino do F4 (dezembro de 2019 a junho de 2021, o treino da Task 1), só para leitura. Não há avaliação nesta secção.

In [33]:
tr = df[df["Start Time"] < pd.Timestamp("2021-07-01")]

cols_l, pipe_l = ab.criar_pipeline("logistica", GRUPOS_TODOS, "posto")
pipe_l.fit(tr[cols_l], tr["is_Abnormal"])
coef = pd.Series(pipe_l.named_steps["modelo"].coef_[0],
                 index=pipe_l.named_steps["pre"].get_feature_names_out())

e_dummy = coef.index.str.startswith(("Charging Post ID", "hour_", "dow_"))
display(coef[~e_dummy].sort_values(key=abs, ascending=False).round(3).to_frame("coeficiente"))
pd.DataFrame({"n.º colunas": [e.sum() for e in [coef.index.str.startswith(p) for p in ("Charging Post ID", "hour_", "dow_")]],
              "amplitude dos coeficientes": [coef[coef.index.str.startswith(p)].agg(lambda s: s.max() - s.min())
                                             for p in ("Charging Post ID", "hour_", "dow_")]},
             index=["posto", "hora", "dia da semana"]).round(3)

,coeficiente
post_previous_abnormal_rate_1D,0.721
post_last_completed_abnormal,0.526
user_last_completed_abnormal,0.493
post_previous_abnormal_1D,-0.489
post_previous_abnormal_rate_7D,0.379
user_previous_sessions_180D,-0.302
user_previous_abnormal_180D,0.244
post_previous_sessions_1D,0.202
post_previous_abnormal_30D,0.200
user_previous_abnormal_rate_30D,0.180


,n.º colunas,amplitude dos coeficientes
posto,92,0.668
hora,24,0.195
dia da semana,7,0.061


In [34]:
from sklearn.tree import export_text

cols_a, pipe_a = ab.criar_pipeline("arvore", GRUPOS_TODOS, "posto", max_depth=12, min_samples_leaf=200)
pipe_a.fit(tr[cols_a], tr["is_Abnormal"])
arv = pipe_a.named_steps["modelo"]
nomes_a = list(pipe_a.named_steps["pre"].get_feature_names_out())

imp = pd.Series(arv.feature_importances_, index=nomes_a)
imp_grupo = imp.groupby(lambda c: "posto (one-hot)" if c.startswith("Charging Post ID") else c).sum()
display(imp_grupo.sort_values(ascending=False).head(12).round(3).to_frame("importância"))
print("folhas:", arv.get_n_leaves(), "| profundidade:", arv.get_depth())
print(export_text(arv, feature_names=nomes_a, max_depth=3, decimals=2))

,importância
post_previous_abnormal_rate_1D,0.522
user_last_completed_abnormal,0.155
post_last_completed_abnormal,0.101
post_previous_abnormal_rate_30D,0.070
post_previous_abnormal_rate_7D,0.047
user_previous_abnormal_rate_180D,0.038
user_previous_abnormal_rate_30D,0.022
post_hours_since_previous_completed_abnormal,0.006
user_days_since_previous_completed_abnormal,0.006
post_previous_sessions_1D,0.004


folhas: 510 | profundidade: 12
|--- post_previous_abnormal_rate_1D <= 0.42
|   |--- user_last_completed_abnormal <= 0.50
|   |   |--- post_previous_abnormal_rate_30D <= 0.17
|   |   |   |--- user_previous_abnormal_rate_180D <= 0.16
|   |   |   |   |--- truncated branch of depth 9
|   |   |   |--- user_previous_abnormal_rate_180D >  0.16
|   |   |   |   |--- truncated branch of depth 9
|   |   |--- post_previous_abnormal_rate_30D >  0.17
|   |   |   |--- post_previous_abnormal_rate_7D <= 0.32
|   |   |   |   |--- truncated branch of depth 9
|   |   |   |--- post_previous_abnormal_rate_7D >  0.32
|   |   |   |   |--- truncated branch of depth 9
|   |--- user_last_completed_abnormal >  0.50
|   |   |--- post_last_completed_abnormal <= 0.50
|   |   |   |--- user_previous_abnormal_rate_180D <= 0.40
|   |   |   |   |--- truncated branch of depth 9
|   |   |   |--- user_previous_abnormal_rate_180D >  0.40
|   |   |   |   |--- truncated branch of depth 7
|   |   |--- post_last_completed_abnorm

**Leitura.** Nos dois modelos, a decisão assenta no histórico recente do posto (taxa de anomalias no último dia e última sessão anómala) e na última sessão do utilizador. Na logística, os coeficientes das variáveis meteorológicas e da tarifa são inferiores a 0,03 em valor absoluto, e alguns pares redundantes têm sinais opostos (taxa a 1 dia +0,721, contagem a 1 dia −0,489), o que confirma que os coeficientes devem ser lidos por grupo. Na árvore, os históricos somam cerca de 98% da importância e o posto não aparece entre as 12 variáveis mais importantes. A árvore final tem 510 folhas, pelo que só os primeiros níveis são legíveis.

## 9. Ensembles

### 9.1 Diversidade dos modelos base

Um ensemble só melhora os modelos que combina se estes errarem em sessões diferentes. Mede-se, em cada fold, a correlação de Spearman entre os scores dos três modelos, a percentagem de alertas em comum e as falhas detetadas por cada modelo e por nenhum dos outros.

In [35]:
from scipy.stats import spearmanr

BASE = ["logistica", "arvore", "naive_bayes (sem posto)"]

linhas = {}
for i, a in enumerate(BASE):
    for b in BASE[i + 1:]:
        cor, comum = [], []
        for f in scores[a]:
            sa, pa = scores[a][f][1], scores[a][f][2].astype(bool)
            sb, pb = scores[b][f][1], scores[b][f][2].astype(bool)
            cor.append(spearmanr(sa, sb)[0])
            comum.append((pa & pb).sum() / pa.sum())
        linhas[(a, b)] = {"correlação de Spearman": np.mean(cor), "% alertas em comum": np.mean(comum)}
display(pd.DataFrame(linhas).T.round(3))

linhas = {}
for f in scores["logistica"]:
    y = scores["logistica"][f][0] == 1
    p = {m: scores[m][f][2].astype(bool) for m in BASE}
    linha = {"detetadas pelos três": (y & p[BASE[0]] & p[BASE[1]] & p[BASE[2]]).sum() / y.sum(),
             "detetadas por pelo menos um": (y & (p[BASE[0]] | p[BASE[1]] | p[BASE[2]])).sum() / y.sum()}
    for m in BASE:
        outros = np.logical_or.reduce([p[o] for o in BASE if o != m])
        linha[f"só {m}"] = (y & p[m] & ~outros).sum() / y.sum()
    linhas[f] = linha
div = pd.DataFrame(linhas).T
div.loc["média"] = div.mean()
div.round(3)

correlação de Spearman  % alertas em comum
logistica arvore                                    0.896               0.832
          naive_bayes (sem posto)                   0.898               0.877
arvore    naive_bayes (sem posto)                   0.805               0.819

,detetadas pelos três,detetadas por pelo menos um,só logistica,só arvore,só naive_bayes (sem posto)
F1,0.522,0.655,0.020,0.028,0.017
F2,0.497,0.632,0.018,0.036,0.014
F3,0.534,0.651,0.015,0.030,0.017
F4,0.499,0.629,0.014,0.029,0.018
média,0.513,0.642,0.017,0.031,0.017


**Leitura.** Os três modelos são muito semelhantes: correlação de Spearman entre 0,80 e 0,90 e 82% a 88% dos alertas em comum. Metade das falhas (51%) é detetada pelos três, e cada modelo deteta sozinho apenas 1,7% (logística e Naive Bayes) a 3,1% (árvore) das falhas. Os três juntos detetam 64% das falhas, mas com até três vezes mais alertas do que o orçamento, pelo que este valor é apenas um limite superior. A combinação dos modelos base tem, por isso, pouca margem de ganho; a árvore é o modelo mais diferente dos outros dois.

### 9.2 Random Forest

Bagging de árvores com seleção aleatória de variáveis em cada divisão: cada árvore é treinada numa amostra bootstrap e só considera uma parte das variáveis em cada nó, o que reduz a correlação entre árvores. A média de muitas árvores pouco correlacionadas reduz a variância, que é o principal defeito de uma árvore isolada (Tabela 42). Usa a combinação 4 sem o posto: com `max_features = "sqrt"`, as 92 colunas one-hot do posto seriam a maioria das variáveis sorteadas em cada nó, e o posto não acrescenta informação quando os históricos estão presentes (secção 3). Configuração inicial: 200 árvores, folha mínima de 100 sessões.

In [36]:
import importlib
importlib.reload(ab)

correr("random_forest", "random_forest", None)
display(tabela(["logistica", "arvore", "naive_bayes (sem posto)", "random_forest"]))
pd.DataFrame({b: comparar(resultados["random_forest"], resultados[b], "recall")
              for b in ["logistica", "arvore"]}).T.round(4)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média),tempo 4 folds (s)
logistica,0.580,0.015,0.566,0.521,0.831,20.983
arvore,0.578,0.013,0.564,0.525,0.823,24.267
naive_bayes (sem posto),0.574,0.017,0.559,0.498,0.814,2.269
random_forest,0.587,0.013,0.576,0.543,0.835,99.636


,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B
logistica,0.0079,0.0020,4.0,0.5762,0.5657
arvore,0.0094,0.0041,4.0,0.5762,0.5639


In [37]:
afin_rf = {"min_samples_leaf=100, max_features=sqrt (inicial)": resultados["random_forest"]}
for leaf in (50, 200):
    for mf in ("sqrt", 0.5):
        afin_rf[f"min_samples_leaf={leaf}, max_features={mf}"] = avaliar_folds(
            df, ab.fabricar_scorer("random_forest", GRUPOS_TODOS, None, min_samples_leaf=leaf, max_features=mf), X_ALERTAS)
display(pd.DataFrame({k: metricas_fold(r) for k, r in afin_rf.items()}).T
        .sort_values("recall@200 (média)", ascending=False).round(3))
base_rf = afin_rf["min_samples_leaf=100, max_features=sqrt (inicial)"]
pd.DataFrame({k: comparar(r, base_rf, "recall") for k, r in afin_rf.items() if "inicial" not in k}).T.round(4)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
"min_samples_leaf=50, max_features=sqrt",0.588,0.013,0.577,0.545,0.835
"min_samples_leaf=50, max_features=0.5",0.588,0.014,0.576,0.547,0.834
"min_samples_leaf=100, max_features=sqrt (inicial)",0.587,0.013,0.576,0.543,0.835
"min_samples_leaf=200, max_features=0.5",0.587,0.014,0.575,0.546,0.835
"min_samples_leaf=200, max_features=sqrt",0.586,0.013,0.574,0.540,0.835


,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B
"min_samples_leaf=50, max_features=sqrt",0.0010,0.0001,4.0,0.5773,0.5762
"min_samples_leaf=50, max_features=0.5",0.0002,0.0009,2.0,0.5759,0.5762
"min_samples_leaf=200, max_features=sqrt",-0.0011,0.0013,1.0,0.5740,0.5762
"min_samples_leaf=200, max_features=0.5",-0.0003,0.0012,1.0,0.5749,0.5762


In [38]:
correr("random_forest", "random_forest", None, min_samples_leaf=50, max_features=0.5)
display(tabela(["logistica", "arvore", "naive_bayes (sem posto)", "random_forest"]))
pd.DataFrame({b: comparar(resultados["random_forest"], resultados[b], "recall")
              for b in ["logistica", "arvore", "naive_bayes (sem posto)"]}).T.round(4)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média),tempo 4 folds (s)
logistica,0.580,0.015,0.566,0.521,0.831,20.983
arvore,0.578,0.013,0.564,0.525,0.823,24.267
naive_bayes (sem posto),0.574,0.017,0.559,0.498,0.814,2.269
random_forest,0.588,0.014,0.576,0.547,0.834,254.658


,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B
logistica,0.0081,0.0014,4.0,0.5759,0.5657
arvore,0.0097,0.0040,4.0,0.5759,0.5639
naive_bayes (sem posto),0.0132,0.0031,4.0,0.5759,0.5593


**Decisão.** A Random Forest supera a logística (+0,0081) e a árvore (+0,0097) nos quatro folds, com melhor pior fold, e é preferida a ambas pela regra da secção 8.9. Na afinação, as cinco configurações ficam entre 0,586 e 0,588 (diferenças inferiores a 0,005, logo empate); o desempate pela PR-AUC normalizada escolhe folha mínima de 50 e max_features = 0,5 (0,547).

### 9.3 Boosting

Gradient boosting com árvores (`HistGradientBoostingClassifier`): cada árvore nova é ajustada aos erros do conjunto das anteriores, o que reduz o viés e permite captar interações entre os históricos que uma árvore isolada ou a logística não captam. Usa a combinação 4 sem o posto, como a Random Forest. O early stopping está desligado, porque por omissão usa uma divisão aleatória do treino, que poria sessões posteriores a decidir sobre anteriores; o número de iterações é escolhido pelos folds temporais.

In [39]:
importlib.reload(ab)

afin_gb = {}
for lr in (0.05, 0.1):
    for it in (200, 500):
        afin_gb[f"learning_rate={lr}, max_iter={it}"] = avaliar_folds(
            df, ab.fabricar_scorer("boosting", GRUPOS_TODOS, None, learning_rate=lr, max_iter=it), X_ALERTAS)
display(pd.DataFrame({k: metricas_fold(r) for k, r in afin_gb.items()}).T
        .sort_values("recall@200 (média)", ascending=False).round(3))
base_gb = afin_gb["learning_rate=0.1, max_iter=200"]
pd.DataFrame({k: comparar(r, base_gb, "recall") for k, r in afin_gb.items()
              if k != "learning_rate=0.1, max_iter=200"}).T.round(4)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média)
"learning_rate=0.05, max_iter=200",0.586,0.014,0.572,0.544,0.835
"learning_rate=0.1, max_iter=200",0.584,0.015,0.569,0.542,0.833
"learning_rate=0.05, max_iter=500",0.584,0.013,0.572,0.541,0.833
"learning_rate=0.1, max_iter=500",0.581,0.013,0.570,0.534,0.829


,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B
"learning_rate=0.05, max_iter=200",0.0022,0.0024,3.0,0.5719,0.5695
"learning_rate=0.05, max_iter=500",-0.0001,0.0018,2.0,0.5717,0.5695
"learning_rate=0.1, max_iter=500",-0.0027,0.0028,1.0,0.5699,0.5695


In [40]:
correr("boosting", "boosting", None, learning_rate=0.05, max_iter=200)
display(tabela(["logistica", "arvore", "naive_bayes (sem posto)", "random_forest", "boosting"]))
pd.DataFrame({b: comparar(resultados["boosting"], resultados[b], "recall")
              for b in ["logistica", "arvore", "naive_bayes (sem posto)", "random_forest"]}).T.round(4)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média),tempo 4 folds (s)
logistica,0.580,0.015,0.566,0.521,0.831,20.983
arvore,0.578,0.013,0.564,0.525,0.823,24.267
naive_bayes (sem posto),0.574,0.017,0.559,0.498,0.814,2.269
random_forest,0.588,0.014,0.576,0.547,0.834,254.658
boosting,0.586,0.014,0.572,0.544,0.835,23.149


,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B
logistica,0.0067,0.0011,4.0,0.5719,0.5657
arvore,0.0082,0.0051,4.0,0.5719,0.5639
naive_bayes (sem posto),0.0118,0.0029,4.0,0.5719,0.5593
random_forest,-0.0015,0.0019,0.0,0.5719,0.5759


**Decisão.** As quatro configurações do boosting ficam entre 0,581 e 0,586 (diferenças inferiores a 0,005, logo empate); o desempate pela PR-AUC normalizada escolhe learning_rate = 0,05 e 200 iterações (0,544). Mais iterações pioram o resultado, sinal de sobreajuste ao treino.

### 9.4 Votação suave (soft voting)

Combina os modelos individuais, de princípios diferentes, pela média dos seus scores em cada sessão. Testam-se duas formas: a média das probabilidades e a média das posições (ranks) de cada sessão na ordenação de cada modelo. A segunda impede que o Naive Bayes, cujas probabilidades são demasiado extremas, domine a média. Os ranks são calculados dentro de cada período de validação, tal como o orçamento de alertas; em operação seriam calculados face aos scores recentes, como o limiar móvel (secção 8.8). Testam-se os três modelos e só a logística com a árvore.

In [41]:
from scipy.stats import rankdata

def combinar(componentes, modo):
    linhas, sc = {}, {}
    for f in scores[componentes[0]]:
        y = scores[componentes[0]][f][0]
        assert all(np.array_equal(scores[m][f][0], y) for m in componentes)
        if modo == "probabilidades":
            s = np.mean([scores[m][f][1] for m in componentes], axis=0)
        else:
            s = np.mean([rankdata(scores[m][f][1]) / len(y) for m in componentes], axis=0)
        pred = alertas_por_orcamento(s, X_ALERTAS)
        linhas[f] = avaliar(y, s, pred=pred)
        sc[f] = (y, s, pred)
    return pd.DataFrame(linhas).T.astype(float), sc

VOTOS = {"voto: logística + árvore + NB": ["logistica", "arvore", "naive_bayes (sem posto)"],
         "voto: logística + árvore": ["logistica", "arvore"]}
for nome_v, comp in VOTOS.items():
    for modo in ("probabilidades", "ranks"):
        n = f"{nome_v} ({modo})"
        resultados[n], scores[n] = combinar(comp, modo)
        tempos[n] = sum(tempos[m] for m in comp)

nomes_v = [n for n in resultados if n.startswith("voto")]
display(tabela(["logistica", "arvore"] + nomes_v))
pd.DataFrame({(n, b): comparar(resultados[n], resultados[b], "recall")
              for n in nomes_v for b in ["logistica", "random_forest"]}).T.round(4)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média),tempo 4 folds (s)
logistica,0.580,0.015,0.566,0.521,0.831,20.983
arvore,0.578,0.013,0.564,0.525,0.823,24.267
voto: logística + árvore + NB (probabilidades),0.580,0.012,0.567,0.535,0.831,47.519
voto: logística + árvore + NB (ranks),0.583,0.013,0.571,0.535,0.831,47.519
voto: logística + árvore (probabilidades),0.583,0.012,0.572,0.536,0.832,45.250
voto: logística + árvore (ranks),0.583,0.013,0.572,0.536,0.832,45.250


diferença média  desvio da diferença  folds em que A ganha  pior fold A  pior fold B
voto: logística + árvore + NB (probabilidades) logistica               0.0006               0.0039                   1.0       0.5671       0.5657
                                               random_forest          -0.0075               0.0025                   0.0       0.5671       0.5759
voto: logística + árvore + NB (ranks)          logistica               0.0039               0.0028                   4.0       0.5712       0.5657
                                               random_forest          -0.0042               0.0020                   0.0       0.5712       0.5759
voto: logística + árvore (probabilidades)      logistica               0.0034               0.0030                   3.0       0.5722       0.5657
                                               random_forest          -0.0047               0.0020                   0.0       0.5722       0.5759
voto: logística + árvore (ranks)               logistica               0.0033               0.0026                   4.0       0.5719       0.5657
                                               random_forest          -0.0048               0.0015                   0.0       0.5719       0.5759

**Decisão.** O boosting supera os três modelos individuais nos quatro folds (+0,007 a +0,012), mas fica abaixo da Random Forest em todos os folds (−0,0015), com um custo cerca de dez vezes menor (23 s contra 255 s). Na votação, a média das probabilidades dos três modelos não supera a logística (melhor em 1 de 4 folds): as probabilidades demasiado extremas do Naive Bayes dominam a média. Com a média dos ranks, ou só com a logística e a árvore, a votação supera a logística em 3 ou 4 folds (+0,003 a +0,004), mas fica abaixo da Random Forest em todos os folds. As três variantes que superam a logística diferem menos de 0,001 entre si; pelo desempate da secção 8.9 (menor desvio), mantém-se a votação da logística e da árvore com média das probabilidades (desvio 0,012, pior fold 0,572).

## 10. Síntese: comparação final

Elegibilidade (secção 8.9), tendência por fold, métricas operacionais e desempenho nas sessões sem histórico do utilizador, para todos os modelos finais.

In [42]:
FINAIS = ["logistica", "arvore", "naive_bayes (sem posto)", "random_forest", "boosting",
          "voto: logística + árvore (probabilidades)"]
linhas = {}
for n in FINAIS:
    r = resultados[n]
    ag = agregar_folds(r, scores[n])
    linhas[n] = {"recall > regra (4 folds)": bool((r["recall"].values > ref_regra_res["recall"].values).all()),
                 "PR-AUC norm. > taxa 7d (4 folds)": bool((r["pr_auc_norm"].values > ref_pr["pr_auc_norm"].values).all()),
                 "tendência recall": ag.loc["recall", "tendência por fold"],
                 "tendência PR-AUC norm.": ag.loc["pr_auc_norm", "tendência por fold"],
                 "precisão": r["precisao"].mean(),
                 "alertas corretos /1000": r["alertas_corretos_1000"].mean(),
                 "falsos alertas /1000": r["falsos_alertas_1000"].mean(),
                 "falhas não detetadas /1000": r["falhadas_1000"].mean(),
                 "recall sem histórico do utilizador": recall_subconjunto(scores[n], mask_sem_hist).mean()}
display(pd.DataFrame(linhas).T.round(3))
pd.DataFrame({**{n: resultados[n]["recall"] for n in FINAIS}, "ref. última sessão": ref_regra_res["recall"]}).round(3)

,recall > regra (4 folds),PR-AUC norm. > taxa 7d (4 folds),tendência recall,tendência PR-AUC norm.,precisão,alertas corretos /1000,falsos alertas /1000,falhas não detetadas /1000,recall sem histórico do utilizador
logistica,True,True,-0.004644,0.017114,0.521437,104.286208,95.71115,75.770357,0.456471
arvore,True,True,-0.003701,0.02165,0.520314,104.061684,95.935674,75.994881,0.405482
naive_bayes (sem posto),True,True,-0.004914,0.008242,0.516823,103.363459,96.633898,76.693106,0.44221
random_forest,True,True,-0.003894,0.019387,0.528847,105.768219,94.229139,74.288346,0.423947
boosting,True,True,-0.005235,0.019353,0.52738,105.474706,94.522652,74.581859,0.411229
voto: logística + árvore (probabilidades),True,True,-0.003879,0.018742,0.524582,104.915145,95.082213,75.14142,0.442907


,logistica,arvore,naive_bayes (sem posto),random_forest,boosting,voto: logística + árvore (probabilidades),ref. última sessão
F1,0.590,0.590,0.586,0.597,0.597,0.592,0.520
F2,0.568,0.564,0.561,0.576,0.576,0.573,0.482
F3,0.594,0.587,0.592,0.602,0.600,0.594,0.526
F4,0.566,0.570,0.559,0.576,0.572,0.572,0.518


**Escolha do modelo final.** Os seis modelos são elegíveis em todos os folds. Pela regra da secção 8.9, a Random Forest (combinação 4 sem posto; 200 árvores, folha mínima 50, max_features = 0,5) é preferida a todos os outros: maior recall médio (0,588), melhor nos quatro folds e melhor pior fold (0,576). Em relação à logística, deteta mais 1,5 falhas por 1000 sessões (105,8 contra 104,3) e, em relação à referência, mais 13,5 (contra 92,3). Ressalvas: (1) o recall de todos os modelos tem tendência ligeiramente negativa (−0,004 a −0,005 por fold, contra +0,004 da referência), sinalizada pela regra da secção 8.9, enquanto a PR-AUC normalizada sobe; (2) nas sessões sem histórico do utilizador, a Random Forest (0,424) fica abaixo da logística (0,456); (3) custa cerca de dez vezes mais a treinar do que o boosting, que fica a 0,0015. O modelo final é retreinado com todo o período de desenvolvimento e avaliado uma única vez no teste, com o limiar móvel de 7 dias (secção 8.8).